In [ ]:
import pickle

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from training_config import DATASET_CONFIGS, FEATURE_COLUMNS, TARGET_COLUMN

In [ ]:
def build_model(model_name):
    if model_name == "linear_regression":
        return LinearRegression()
    if model_name == "random_forest":
        return RandomForestRegressor(n_estimators=200, random_state=42)
    raise ValueError(f"Unsupported model: {model_name}")


def prepare_dataset(path):
    df = pd.read_csv(path)
    df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
    df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})
    df = df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN])
    return df


In [ ]:
trained_models = {}
dataset_results = []

for config in DATASET_CONFIGS:
    dataset = prepare_dataset(config["path"])
    X = dataset[FEATURE_COLUMNS]
    y = dataset[TARGET_COLUMN]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = build_model(config["model_name"])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    trained_models[config["name"]] = model
    dataset_results.append(
        {
            "Dataset": config["name"],
            "Source_File": config["path"],
            "Algorithm": config["model_name"],
            "Rows_Used": len(dataset),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R2": r2_score(y_test, predictions),
        }
    )

results_df = pd.DataFrame(dataset_results).sort_values("R2", ascending=False)
results_df

In [ ]:
def ensemble_predict(model_bundle, input_frame):
    predictions = []
    for model in model_bundle["models"].values():
        predictions.append(model.predict(input_frame))
    prediction_frame = pd.DataFrame(predictions).T
    return prediction_frame.mean(axis=1).values


model_bundle = {
    "model_type": "multi_dataset_ensemble",
    "feature_columns": FEATURE_COLUMNS,
    "target_column": TARGET_COLUMN,
    "datasets": DATASET_CONFIGS,
    "models": trained_models,
}


In [ ]:
sample_dataset = prepare_dataset(DATASET_CONFIGS[0]["path"])
sample_inputs = sample_dataset[FEATURE_COLUMNS].head().copy()
sample_outputs = sample_dataset[[TARGET_COLUMN]].head().copy()
sample_outputs["Ensemble_Prediction"] = ensemble_predict(model_bundle, sample_inputs)
pd.concat([sample_inputs, sample_outputs], axis=1)

In [ ]:
pickle.dump(model_bundle, open("model.pkl", "wb"))
print("Required dataset models trained and saved to model.pkl")